# Part 2.7 Subagents

After this notebook you can hand bulky work to a subagent that runs in its own context and
returns a short answer.

In [ ]:
from kea.config import MODEL, setup
from kea.notebook import ask_and_approve, final, steps

setup("kea-rebooking-local")

## See it

The agent hands the policy question to a `policy-checker` subagent with a `task` call. The
subagent reads the policy in its own context and sends back a short answer.

<img src="../../assets/screenshots/p2-07-subagent-trace.png" alt="the trace of a policy question: the task call, with the policy-checker run expanded underneath and its readfile of the skill." width="900">

## Build it

Reading the whole policy puts a page of text in the agent's context. A **subagent** does
the reading in its own context and sends back a short answer. A subagent is a dict: a
name, a description the parent chooses by, its own prompt, its own tools, and its own
skills.

In [ ]:
from deepagents import create_deep_agent

from kea.rebooking.agent import build_backend

QUESTION = (
    "KA 1182 was cancelled for a mechanical problem, and the next flight I can take "
    "leaves 145 minutes later. What credit am I owed?"
)

policy_checker = {
    "name": "policy-checker",
    "description": (
        "Reads Kea Air's rebooking policy and says what a passenger is entitled to. "
        "Give it the cancellation cause, the delay in minutes, and what the passenger "
        "asked for."
    ),
    "system_prompt": (
        "Read the rebooking-policy skill, then answer in five lines or fewer: the "
        "entitlement, the amount, whether supervisor approval is needed, and anything "
        "to decline."
    ),
    "tools": [],
    "skills": ["/skills/"],
}

delegating_agent = create_deep_agent(
    model=MODEL,
    system_prompt=(
        "You are Kea Air's rebooking assistant. Ask the policy-checker what a passenger "
        "is owed. Do not read the policy yourself."
    ),
    backend=build_backend(),
    subagents=[policy_checker],
)

result = await ask_and_approve(delegating_agent, QUESTION)
steps(result)
print()
final(result)

In [ ]:
print("policy text in the parent's context:", "Entitlements" in str(result["messages"]))

## Under the hood

- **Context isolation.** A subagent runs its own loop in a separate context and returns
  only its last message. The parent sees one tool result: not the skill, not the reads.
  The cost is that if the parent later needs a detail the subagent saw but didn't return,
  it's gone. Tell the subagent what to return.
- **The finished agent also has the skill itself,** so its parent sometimes reads the
  policy directly. The Build it agent has no skills, so the check above is clean.
- **Tools are inherited unless you set them.** A subagent without a `tools` key gets the
  parent's tools. We set `tools: []` so the `policy-checker` can't issue credit.
- **Each `task` is a whole extra loop:** more latency, more tokens, one more place to
  garble the request. Don't delegate a short task, work that needs the whole conversation,
  or output that is hard to summarize. Delegate work that is bulky, separable, and easy to
  summarize.
- Deep Agents also gives you a general-purpose subagent through `task` with no
  configuration.

Docs: [Deep Agents subagents](https://docs.langchain.com/oss/python/deepagents/subagents).

## Recap

- A subagent is a name, a description, a prompt, tools, and skills.
- It works in its own context and returns only a short answer, so the parent's context
  stays small.
- Set `tools` on purpose. Delegate work that is bulky, separable, and easy to summarize.

**Next:** [Part 2.8 Middleware](08_middleware.ipynb)